In [44]:
import os
print(os.listdir("dataset"))

['fake', 'real']


In [1]:
import os

path = r"I:\Research\Deepfake Detection program\dataset"

print("Exists:", os.path.exists(path))
print("Folders:", os.listdir(path))

print("Real count:", len(os.listdir(os.path.join(path, "real"))))
print("Fake count:", len(os.listdir(os.path.join(path, "fake"))))

Exists: True
Folders: ['fake', 'real']
Real count: 1000
Fake count: 1000


In [1]:
import os
import shutil
import cv2
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader, random_split

# ==============================
# 1. SAFE DATASET PREPARATION
# ==============================
source = r"I:\Research\Deepfake Detection program\archive (1)\FaceForensics++_c23"
target = r"I:\Research\Deepfake Detection program\dataset"

real_path = os.path.join(target, "real")
fake_path = os.path.join(target, "fake")

# Only create dataset if not already exists
if not os.path.exists(real_path) or not os.path.exists(fake_path):
    print("Preparing dataset...")

    os.makedirs(real_path, exist_ok=True)
    os.makedirs(fake_path, exist_ok=True)

    # Copy real
    for file in os.listdir(os.path.join(source, "original")):
        shutil.copy(os.path.join(source, "original", file), real_path)

    # Copy fake
    fake_folders = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures", "FaceShifter"]
    for folder in fake_folders:
        folder_path = os.path.join(source, folder)
        for file in os.listdir(folder_path):
            shutil.copy(os.path.join(folder_path, file), fake_path)

    print("Dataset ready!")
else:
    print("Dataset already exists, skipping copy.")

# ==============================
# 2. FRAME EXTRACTION
# ==============================
def extract_frames(video_path, num_frames=5):
    cap = cv2.VideoCapture(video_path)
    frames = []

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(total // num_frames, 1)

    for i in range(num_frames):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i * step)
        ret, frame = cap.read()

        if ret:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (224, 224))

            frame = frame / 255.0
            frame = (frame - np.array([0.485,0.456,0.406])) / np.array([0.229,0.224,0.225])

            frame = np.transpose(frame, (2,0,1))
            frames.append(frame)

    cap.release()

    if len(frames) == 0:
        return torch.zeros((num_frames, 3, 224, 224))

    while len(frames) < num_frames:
        frames.append(frames[-1])

    return torch.from_numpy(np.array(frames).astype(np.float32))

# ==============================
# 3. DATASET
# ==============================
class VideoDataset(Dataset):
    def __init__(self, root_dir):
        self.samples = []

        for label, folder in enumerate(['real', 'fake']):
            path = os.path.join(root_dir, folder)

            if not os.path.exists(path):
                continue

            for file in os.listdir(path):
                if file.lower().endswith(".mp4"):
                    self.samples.append((os.path.join(path, file), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        frames = extract_frames(video_path)
        return frames, torch.tensor([label], dtype=torch.float32)

# ==============================
# 4. MODEL
# ==============================
class AttentionBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // 8),
            nn.ReLU(),
            nn.Linear(channels // 8, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, h, w = x.size()
        avg_pool = torch.mean(x, dim=[2,3])
        attention = self.fc(avg_pool).view(b, c, 1, 1)
        return x * attention

class FFTBranch(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )

    def forward(self,x):
        gray = torch.mean(x, dim=1, keepdim=True)
        fft = torch.fft.fft2(gray)
        fft = torch.log(torch.abs(fft)+1e-6)
        features = self.conv(fft)
        return features.view(features.size(0),-1)

class DeepfakeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.resnet.fc = nn.Identity()

        self.attention = AttentionBlock(2048)
        self.fft_branch = FFTBranch()

        self.fusion = nn.Sequential(
            nn.Linear(2048+32,512),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

        self.classifier = nn.Sequential(
            nn.Linear(512,64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64,1),
            nn.Sigmoid()
        )

    def forward(self,x):
        b,t,c,h,w = x.shape
        frame = x[:,0]

        spatial = self.resnet(frame)
        spatial = spatial.view(b,2048,1,1)
        spatial = self.attention(spatial).view(b,-1)

        freq = self.fft_branch(frame)

        fused = self.fusion(torch.cat([spatial,freq],dim=1))
        return self.classifier(fused)

# ==============================
# 5. TRAINING WITH ACCURACY
# ==============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

dataset = VideoDataset(target)
print("Total videos:", len(dataset))

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

model = DeepfakeDetector().to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

# Function to calculate accuracy
def calculate_accuracy(outputs, labels):
    predicted = (outputs > 0.5).float()
    correct = (predicted == labels).sum().item()
    total = labels.size(0)
    return correct / total

# Function to validate
def validate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for videos, labels in val_loader:
            videos, labels = videos.to(device), labels.to(device)
            outputs = model(videos)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            predicted = (outputs > 0.5).float()
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

    avg_loss = total_loss / len(val_loader)
    avg_accuracy = total_correct / total_samples if total_samples > 0 else 0
    return avg_loss, avg_accuracy

# Training loop
for epoch in range(50):
    model.train()
    train_loss = 0
    train_correct = 0
    train_samples = 0

    for videos, labels in train_loader:
        videos, labels = videos.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(videos)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        predicted = (outputs > 0.5).float()
        train_correct += (predicted == labels).sum().item()
        train_samples += labels.size(0)

    # Calculate training metrics
    avg_train_loss = train_loss / len(train_loader)
    train_accuracy = train_correct / train_samples if train_samples > 0 else 0

    # Validate
    val_loss, val_accuracy = validate(model, val_loader, criterion, device)

    print(f"Epoch {epoch+1}/50 | Train Loss: {avg_train_loss:.4f} | Train Acc: {train_accuracy:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.4f}")

torch.save(model.state_dict(), "deepfake_model_final.pth")
print("Training complete! Model saved as deepfake_model_final.pth")


Dataset already exists, skipping copy.
Device: cuda
Total videos: 2000
Epoch 1/50 | Train Loss: 0.6891 | Train Acc: 0.5294 | Val Loss: 0.7439 | Val Acc: 0.5950
Epoch 2/50 | Train Loss: 0.6521 | Train Acc: 0.6369 | Val Loss: 0.5342 | Val Acc: 0.7425
Epoch 3/50 | Train Loss: 0.6607 | Train Acc: 0.5787 | Val Loss: 0.6954 | Val Acc: 0.5025
Epoch 4/50 | Train Loss: 0.6799 | Train Acc: 0.5700 | Val Loss: 0.8307 | Val Acc: 0.4700
Epoch 5/50 | Train Loss: 0.6961 | Train Acc: 0.4938 | Val Loss: 0.7048 | Val Acc: 0.4700
Epoch 6/50 | Train Loss: 0.6640 | Train Acc: 0.5906 | Val Loss: 0.7531 | Val Acc: 0.6525
Epoch 7/50 | Train Loss: 0.6242 | Train Acc: 0.6787 | Val Loss: 0.5527 | Val Acc: 0.6650
Epoch 8/50 | Train Loss: 0.5843 | Train Acc: 0.7113 | Val Loss: 0.5229 | Val Acc: 0.7375
Epoch 9/50 | Train Loss: 0.5261 | Train Acc: 0.7600 | Val Loss: 0.4851 | Val Acc: 0.7925
Epoch 10/50 | Train Loss: 0.4972 | Train Acc: 0.7850 | Val Loss: 0.4747 | Val Acc: 0.7750
Epoch 11/50 | Train Loss: 0.4647 | Tra